# VSCS 2D component masks on Colab (P2-T2)

Grounding DINO finds each component on keyframes; SAM 2 tracks the masks through the whole scan. GPU stage, so it runs here.

> **No logic here** (CLAUDE.md §4.3): install, mount, call `scripts/seg.py`.

**Before running:** upload the scan's extracted frames folder (from `scripts/extract_frames.py`) to Drive.

**Licences** (§10): Grounding DINO and SAM 2 are Apache-2.0, code and weights. Confirm on first run and record the working library versions in an ADR (R-12).

**Budget** (§1): T4. Log the runtime and compute units in today's devlog (R-11).

## 1. GPU check

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. Mount Drive and point at the frames

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

VSCS_DRIVE = Path("/content/drive/MyDrive/vscs")
# Edit to the frames folder you uploaded:
FRAMES = VSCS_DRIVE / "data/processed/capture/<YYYYMMDD-HHMM_sha>/frames"
OUT_ROOT = VSCS_DRIVE / "data/processed"
print(FRAMES, "OK" if FRAMES.exists() else "MISSING")

## 3. Get the code and install the models

Colab's own torch is kept: nothing here installs or pins torch (R-12).

`hf_xet` is needed to download the Grounding DINO weights from Hugging Face (found in the CPU check, 2026-09-29).

In [ ]:
import os

REPO_URL = "https://github.com/AlvinKalaivannan/VSCS-Vehicle-Scanning-Collision-System-.git"
REPO_DIR = Path("/content/vscs")
if not REPO_DIR.exists():
    !git clone --branch main {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
!pip install --quiet pyyaml "pydantic>=2" numpy opencv-python transformers hf_xet pillow

In [ ]:
SAM2_DIR = Path("/content/sam2")
if not SAM2_DIR.exists():
    !git clone https://github.com/facebookresearch/sam2.git {SAM2_DIR}
!pip install --quiet -e {SAM2_DIR}
!cd {SAM2_DIR}/checkpoints && bash download_ckpts.sh

## 4. Run the masks stage

In [ ]:
!python scripts/seg.py --frames "{FRAMES}" --sam2-dir "{SAM2_DIR}" --out-root "{OUT_ROOT}"

## 5. Before closing the tab

- [ ] The seg run folder (`labels/`, `check/`, `masks_summary.json`) is on **Drive**.
- [ ] Look at every image in `check/` (the P2-T2 visual check). Log failures in today's devlog, including any component listed as never detected.
- [ ] Runtime and compute units noted in the devlog (R-11).
- [ ] Working `transformers` / `sam2` versions recorded in an ADR (R-12).